In [ ]:
import numpy as np
import pandas as pd

file_path = r"C:\Users\USER\Downloads\loan.csv"
df = pd.read_csv(file_path)

print("RAW DATASET")
print(f"intial dimensions: {df.shape[0]}rows, {df.shape[1]}columns")
df.columns = df.columns.str.strip().str.lower()
print(f"Normalized Column Headers: {list(df.columns)}\n")

print("HANDLING MISSING VALUES")

print(df.isnull().sum())
df['loanamount'] = df['loanamount'].fillna(df['loanamount'].median())
df['loan_amount_term'] = df['loan_amount_term'].fillna(df['loan_amount_term'].median())

categorical_cols = ['gender', 'married', 'dependents', 'self_employed', 'credit_history']
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])
print(df.isnull().sum())
df['applicantincome'] = df['applicantincome'].astype(float)
df['coapplicantincome'] = df['coapplicantincome'].astype(float)
df['loanamount'] = df['loanamount'].astype(float)
df['loan_amount_term'] = df['loan_amount_term'].astype(int)

print("COMPREHENSIVE STATISTICAL ANALYSIS")
numeric_features = ['applicantincome', 'coapplicantincome', 'loanamount', 'loan_amount_term']

for col in numeric_features:
    series = df[col]
    mean_val = series.mean()
    median_val = series.median()
    std_val = series.std()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    print(f"Feature Column: [{col.upper()}]")
    print(f"  Mean : {mean_val:.2f}")
    print(f"  Median : {median_val:.2f}")
    print(f"  Standard Deviation: {std_val:.2f}")
    print(f"  First Quartile : {q1:.2f}")
    print(f"  Third Quartile : {q3:.2f}")
    print(f"  IQR : {iqr:.2f}")
    print(f"  Lower Outlier: {lower_bound:.2f}")
    print(f"  Upper Outlier: {upper_bound:.2f}")
    
filename = "capstone_clean.csv"
df.to_csv(filename, index=False)
print(f"Cleaned database stored as: '{filename}'")

RAW DATASET
intial dimensions: 614rows, 13columns
Normalized Column Headers: ['loan_id', 'gender', 'married', 'dependents', 'education', 'self_employed', 'applicantincome', 'coapplicantincome', 'loanamount', 'loan_amount_term', 'credit_history', 'property_area', 'loan_status']

HANDLING MISSING VALUES
loan_id               0
gender               13
married               3
dependents           15
education             0
self_employed        32
applicantincome       0
coapplicantincome     0
loanamount           22
loan_amount_term     14
credit_history       50
property_area         0
loan_status           0
dtype: int64
loan_id              0
gender               0
married              0
dependents           0
education            0
self_employed        0
applicantincome      0
coapplicantincome    0
loanamount           0
loan_amount_term     0
credit_history       0
property_area        0
loan_status          0
dtype: int64
COMPREHENSIVE STATISTICAL ANALYSIS
Feature Column: [APPLICAN

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

df['loan_status'] = df['loan_status'].map({'Y': 1, 'N': 0})

# using metrics as features
X = df[['applicantincome', 'coapplicantincome', 'loanamount', 'loan_amount_term', 'credit_history']]
y = df['loan_status']

# Train/Test Splitting 
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling 
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

# Choose Model & Fit 
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(x_train_scaled, y_train)

# Predict
y_pred = knn_model.predict(x_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"KNN Accuracy : {accuracy:.4f}")
print(f"F1-Score : {f1:.4f} ")
print(classification_report(y_test, y_pred, target_names=['Rejected (0)', 'Approved (1)']))

KNN Accuracy : 0.7724
F1-Score : 0.8444 
              precision    recall  f1-score   support

Rejected (0)       0.83      0.44      0.58        43
Approved (1)       0.76      0.95      0.84        80

    accuracy                           0.77       123
   macro avg       0.79      0.70      0.71       123
weighted avg       0.78      0.77      0.75       123



In [6]:
import pandas as pd
import numpy as np

def predict_new(model, scaler, feature_dict):
    raw_input_df = pd.DataFrame([feature_dict])
    ordered_input_df = raw_input_df[['applicantincome', 'coapplicantincome', 'loanamount', 'loan_amount_term', 'credit_history']]
    scaled_input = scaler.transform(ordered_input_df)
    numeric_prediction = model.predict(scaled_input)[0]
    if numeric_prediction == 1:
        return "LOAN STATUS: APPROVED"
    else:
        return "LOAN STATUS: REJECTED"

applicant_1 = {
    'applicantincome': 7500.0,
    'coapplicantincome': 2500.0,
    'loanamount': 120.0,
    'loan_amount_term': 360,
    'credit_history': 1.0
}

applicant_2 = {
    'applicantincome': 2200.0,
    'coapplicantincome': 0.0,
    'loanamount': 350.0,
    'loan_amount_term': 360,
    'credit_history': 0.0
}

applicant_3 = {
    'applicantincome': 4000.0,
    'coapplicantincome': 3000.0,
    'loanamount': 150.0,
    'loan_amount_term': 180,
    'credit_history': 1.0
}

print(predict_new(knn_model, scaler, applicant_1))
print(predict_new(knn_model, scaler, applicant_2))
print(predict_new(knn_model, scaler, applicant_3))

LOAN STATUS: APPROVED
LOAN STATUS: REJECTED
LOAN STATUS: APPROVED
